# Prompt Profile Comparison

Compare prompt profiles and models side-by-side to understand:
- **Output quality**: How do different profiles change the response style?
- **Token usage**: Which profiles are more verbose?
- **Latency**: Does profile complexity affect response time?
- **Model differences**: How do GPT vs Ollama handle different profiles?



In [ ]:
import sys
import time
from pathlib import Path
from typing import Dict, List, Any

import pandas as pd
from dotenv import load_dotenv

# Add project root to path (works in both Jupyter and Python scripts)
# In Jupyter, use current working directory; in scripts, use __file__
try:
    # Try to get path from __file__ (works in Python scripts)
    project_root = Path(__file__).parent.parent
except NameError:
    # In Jupyter notebooks, __file__ doesn't exist, so use current directory
    # Assume notebook is in experiments/ subdirectory
    project_root = Path.cwd()
    # If we're in the notebook directory, go up two levels
    if project_root.name == "experiments":
        project_root = project_root.parent
    elif (project_root / "experiments" / "prompt_comparison.ipynb").exists():
        # We're in the project root
        pass
    else:
        # Try to find the project root by looking for core/ directory
        current = Path.cwd()
        while current != current.parent:
            if (current / "core").exists() and (current / "prompts").exists():
                project_root = current
                break
            current = current.parent

sys.path.insert(0, str(project_root))

from core.orchestrator import Orchestrator

load_dotenv(override=True)

# Initialize orchestrator
orchestrator = Orchestrator()
orchestrator.load_prompts(str(project_root / "prompts"))

print("Orchestrator initialized")
print(f"Available models: {orchestrator.model_registry.get_available()}")
print(f"Available profiles: {orchestrator.prompt_profiles.available()}")


[INFO] core.model_registry: Validating models...
[INFO] core.model_registry: GPT-4o-mini: Ready
[INFO] core.model_registry: GPT-4o: Ready
[WARNING] core.model_registry: DeepSeek: Invalid API key
[WARNING] core.model_registry: Gemini-Flash: Rate limited (key is valid)
[INFO] core.model_registry: Groq-Llama: Ready
[INFO] core.model_registry: Ollama: Ready
[INFO] core.model_registry: 4/6 models ready
[INFO] core.model_registry: Model Registry Status
[INFO] core.model_registry: GPT-4o-mini: Validated and ready
[INFO] core.model_registry: GPT-4o: Validated and ready
[WARNING] core.model_registry: DeepSeek: Invalid API key
[WARNING] core.model_registry: Gemini-Flash: Rate limited - try again later
[INFO] core.model_registry: Groq-Llama: Validated and ready
[INFO] core.model_registry: Ollama: Validated and ready


Orchestrator initialized
Available models: ['GPT-4o-mini', 'GPT-4o', 'Groq-Llama', 'Ollama']
Available profiles: ['concise_expert', 'teaching_mode', 'reviewer_mode']


import sys
import time
from pathlib import Path
from typing import Dict, List, Any

import pandas as pd
from dotenv import load_dotenv

# Add project root to path (works in both Jupyter and Python scripts)
# In Jupyter, use current working directory; in scripts, use __file__
try:
    # Try to get path from __file__ (works in Python scripts)
    project_root = Path(__file__).parent.parent
except NameError:
    # In Jupyter notebooks, __file__ doesn't exist, so use current directory
    # Assume notebook is in experiments/ subdirectory
    project_root = Path.cwd()
    # If we're in the notebook directory, go up two levels
    if project_root.name == "experiments":
        project_root = project_root.parent
    elif (project_root / "experiments" / "prompt_comparison.ipynb").exists():
        # We're in the project root
        pass
    else:
        # Try to find the project root by looking for core/ directory
        current = Path.cwd()
        while current != current.parent:
            if (current / "core").exists() and (current / "prompts").exists():
                project_root = current
                break
            current = current.parent

sys.path.insert(0, str(project_root))

from core.orchestrator import Orchestrator

load_dotenv(override=True)

# Initialize orchestrator
orchestrator = Orchestrator()
orchestrator.load_prompts(str(project_root / "prompts"))

print("Orchestrator initialized")
print(f"Available models: {orchestrator.model_registry.get_available()}")
print(f"Available profiles: {orchestrator.prompt_profiles.available()}")



In [2]:
def run_comparison(
    test_input: str,
    models: List[str],
    profiles: List[str],
    history: List[Dict[str, str]] = None
) -> pd.DataFrame:
    """
    Run the same input through multiple models and profiles.
    
    Returns DataFrame with columns: model, profile, response, latency, tokens, cost
    """
    results = []
    
    for model_name in models:
        if not orchestrator.model_registry.is_available(model_name):
            print(f"[SKIP] {model_name}: Not available")
            continue
        
        for profile_name in profiles:
            print(f"\n[TEST] {model_name} + {profile_name}")
            
            start_time = time.time()
            
            # Build messages with system prompt from profile
            system_prompt = orchestrator.prompt_profiles.build_system_prompt(profile_name)
            messages = [{"role": "system", "content": system_prompt}]
            messages.extend(history or [])
            messages.append({"role": "user", "content": test_input})
            
            # Make API call
            if orchestrator.model_registry.supports_tools(model_name):
                result = orchestrator.model_registry.chat_with_tools(
                    model_name, messages, None, allow_fallback=False
                )
                if result and result.success:
                    response_obj = result.response
                    content = result.content or ""
                else:
                    print(f"  [ERROR] {result.error if result else 'No response'}")
                    continue
            else:
                # Non-streaming call for measurement
                entry = orchestrator.model_registry.get(model_name)
                response_obj = entry.client.chat.completions.create(
                    model=entry.model,
                    messages=messages,
                    stream=False
                )
                content = response_obj.choices[0].message.content or ""
            
            latency = time.time() - start_time
            
            # Extract token usage
            input_tokens = 0
            output_tokens = 0
            total_tokens = 0
            cost = None
            
            if hasattr(response_obj, 'usage') and response_obj.usage:
                input_tokens = response_obj.usage.prompt_tokens or 0
                output_tokens = response_obj.usage.completion_tokens or 0
                total_tokens = response_obj.usage.total_tokens or 0
                
                # Estimate cost for OpenAI models (gpt-4o-mini pricing)
                if model_name.startswith("GPT"):
                    input_cost = (input_tokens / 1_000_000) * 0.15
                    output_cost = (output_tokens / 1_000_000) * 0.60
                    cost = input_cost + output_cost
            
            results.append({
                "model": model_name,
                "profile": profile_name,
                "response": content,
                "response_length": len(content),
                "latency_sec": round(latency, 2),
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens,
                "cost_usd": round(cost, 6) if cost else None,
            })
            
            print(f"  [OK] {len(content)} chars, {latency:.2f}s, {total_tokens} tokens")
    
    return pd.DataFrame(results)

# Test input
test_input = "Explain what a Python decorator does and show a simple example."

print(f"Test input: {test_input}\n")
print("Running comparison...")


Test input: Explain what a Python decorator does and show a simple example.

Running comparison...


In [3]:
# Run comparison across all available models and profiles
available_models = orchestrator.model_registry.get_available()
available_profiles = orchestrator.prompt_profiles.available()

if not available_models:
    print("No models available. Check your API keys.")
else:
    df = run_comparison(
        test_input=test_input,
        models=available_models,
        profiles=available_profiles
    )
    
    print(f"\n{'='*60}")
    print(f"Comparison Results: {len(df)} runs")
    print(f"{'='*60}\n")
    
    # Display summary table
    summary = df[["model", "profile", "response_length", "latency_sec", "total_tokens", "cost_usd"]].copy()
    print(summary.to_string(index=False))



[TEST] GPT-4o-mini + concise_expert
  [OK] 696 chars, 5.76s, 226 tokens

[TEST] GPT-4o-mini + teaching_mode
  [OK] 3029 chars, 23.32s, 734 tokens

[TEST] GPT-4o-mini + reviewer_mode
  [OK] 1499 chars, 7.80s, 373 tokens

[TEST] GPT-4o + concise_expert
  [OK] 782 chars, 1.97s, 235 tokens

[TEST] GPT-4o + teaching_mode
  [OK] 2801 chars, 5.78s, 661 tokens

[TEST] GPT-4o + reviewer_mode
  [OK] 2213 chars, 4.70s, 543 tokens

[TEST] Groq-Llama + concise_expert
  [OK] 965 chars, 0.85s, 295 tokens

[TEST] Groq-Llama + teaching_mode
  [OK] 2467 chars, 1.34s, 625 tokens

[TEST] Groq-Llama + reviewer_mode
  [OK] 2719 chars, 1.47s, 642 tokens

[TEST] Ollama + concise_expert
  [OK] 1629 chars, 5.02s, 443 tokens

[TEST] Ollama + teaching_mode
  [OK] 2540 chars, 4.30s, 613 tokens

[TEST] Ollama + reviewer_mode
  [OK] 2212 chars, 3.83s, 549 tokens

Comparison Results: 12 runs

      model        profile  response_length  latency_sec  total_tokens  cost_usd
GPT-4o-mini concise_expert              696 

In [4]:
# Detailed comparison: Show responses side-by-side
if len(df) > 0:
    print("\n" + "="*80)
    print("RESPONSE COMPARISON")
    print("="*80 + "\n")
    
    for idx, row in df.iterrows():
        print(f"[{row['model']} + {row['profile']}]")
        print(f"Tokens: {row['total_tokens']} | Latency: {row['latency_sec']}s | Cost: ${row['cost_usd']:.6f}" if row['cost_usd'] else f"Tokens: {row['total_tokens']} | Latency: {row['latency_sec']}s")
        print("-" * 80)
        print(row['response'][:500] + ("..." if len(row['response']) > 500 else ""))
        print("\n")



RESPONSE COMPARISON

[GPT-4o-mini + concise_expert]
Tokens: 226 | Latency: 5.76s | Cost: $0.000099
--------------------------------------------------------------------------------
**Root Cause:** Python decorators allow you to modify or enhance functions or methods at definition time.

**Fix:** A decorator is a function that takes another function as an argument and extends its behavior without explicitly modifying it.

**Example:**
```python
def my_decorator(func):
    def wrapper():
        print("Before the function call.")
        func()
        print("After the function call.")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```

*...


[GPT-4o-mini + teaching_mode]
Tokens: 734 | Latency: 23.32s | Cost: $0.000403
--------------------------------------------------------------------------------
A Python decorator is a special type of function that is used to modify or enhance the behavior of another function or method. Decorators allow you to wra

In [5]:
# Analysis: Compare profiles across models
if len(df) > 0:
    print("\n" + "="*80)
    print("PROFILE ANALYSIS")
    print("="*80 + "\n")
    
    # Group by profile
    for profile in available_profiles:
        profile_df = df[df['profile'] == profile]
        if len(profile_df) > 0:
            print(f"\n{profile.upper()}:")
            print(f"  Avg response length: {profile_df['response_length'].mean():.0f} chars")
            print(f"  Avg latency: {profile_df['latency_sec'].mean():.2f}s")
            print(f"  Avg tokens: {profile_df['total_tokens'].mean():.0f}")
            if profile_df['cost_usd'].notna().any():
                print(f"  Avg cost: ${profile_df['cost_usd'].mean():.6f}")
    
    # Group by model
    print("\n" + "-"*80)
    print("MODEL ANALYSIS")
    print("-"*80 + "\n")
    
    for model in available_models:
        model_df = df[df['model'] == model]
        if len(model_df) > 0:
            print(f"\n{model}:")
            print(f"  Avg response length: {model_df['response_length'].mean():.0f} chars")
            print(f"  Avg latency: {model_df['latency_sec'].mean():.2f}s")
            print(f"  Avg tokens: {model_df['total_tokens'].mean():.0f}")
            if model_df['cost_usd'].notna().any():
                print(f"  Avg cost: ${model_df['cost_usd'].mean():.6f}")



PROFILE ANALYSIS


CONCISE_EXPERT:
  Avg response length: 1018 chars
  Avg latency: 3.40s
  Avg tokens: 300
  Avg cost: $0.000102

TEACHING_MODE:
  Avg response length: 2709 chars
  Avg latency: 8.69s
  Avg tokens: 658
  Avg cost: $0.000381

REVIEWER_MODE:
  Avg response length: 2161 chars
  Avg latency: 4.45s
  Avg tokens: 527
  Avg cost: $0.000240

--------------------------------------------------------------------------------
MODEL ANALYSIS
--------------------------------------------------------------------------------


GPT-4o-mini:
  Avg response length: 1741 chars
  Avg latency: 12.29s
  Avg tokens: 444
  Avg cost: $0.000230

GPT-4o:
  Avg response length: 1932 chars
  Avg latency: 4.15s
  Avg tokens: 480
  Avg cost: $0.000252

Groq-Llama:
  Avg response length: 2050 chars
  Avg latency: 1.22s
  Avg tokens: 521

Ollama:
  Avg response length: 2127 chars
  Avg latency: 4.38s
  Avg tokens: 535


## Insights

**Key Questions to Answer:**
1. Which profile produces the most useful responses for technical questions?
2. How much do profiles affect token usage and cost?
3. Do different models respond differently to the same profile?
4. Is there a latency difference between profiles?

**Next Steps:**
- Try different test inputs (code review, error explanation, document summarization)
- Compare across more models if available
- Measure quality subjectively or with evaluation metrics
